# Session 7: one question, one chart

A chart is an answer to a question. If you cannot say the question in one
sentence, the chart is not ready to draw.

This notebook is the first half of the session: how a figure is built, how
to pick the chart from the question, how a chart can mislead with every
number right, and how to save one. Notebook 02 does time, many charts at
once, highlighting and a figure worth sharing.

Everything here uses the clean file, so the charts never wait on cleaning.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

plays = pd.read_csv("data/clean/plays.csv")
print(len(plays), "plays")

`matplotlib.pyplot` is imported as `plt` by everybody, the same way pandas
is `pd`. It is the library almost every Python chart is built on, pandas'
own `.plot()` included.

---

# The anatomy of a figure

Every chart in this course starts with the same line:

```python
fig, ax = plt.subplots()
```

It hands you two things:

* **`fig`**, the figure: the whole picture, the sheet of paper. You save a
  figure, and a figure can have a title over everything
* **`ax`**, the axes: one chart on that sheet, with its own x-axis, y-axis,
  title and bars. You draw on an `ax`

The name is confusing, so say it once: "axes" means one chart, not the two
lines along its edges. Those are `ax.xaxis` and `ax.yaxis`, and you will
hardly ever touch them.

In [ ]:
device_counts = plays["device"].value_counts()

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(device_counts.index, device_counts.values)
ax.set_title("ax.set_title: the title of this one chart")
ax.set_ylabel("plays")
fig.suptitle("fig.suptitle: a title over the whole figure")

plt.show()

print(type(fig).__name__, type(ax).__name__)

`figsize=(7, 3.5)` is width and height in inches. Most charts here are
wider than they are tall, because labels run sideways.

## Two ways of writing the same chart

Search for any matplotlib question and half the answers look like this:

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.bar(device_counts.index, device_counts.values)
plt.title("the same chart, written with plt.")
plt.ylabel("plays")
plt.show()

That is the **`plt.` style**. Every call goes to "the current chart",
whichever one matplotlib last made. It is short, and fine for one quick
chart.

The **`ax.` style** says which chart, every time. With one chart there is
no difference. With two side by side, "the current chart" is a guess, and
the guess is wrong half the time. This course always writes `ax.`, so
nothing changes when a second chart arrives.

Translating an answer you found online is mostly adding `set_`:

| `plt.` style | `ax.` style |
|---|---|
| `plt.bar(...)`, `plt.plot(...)` | `ax.bar(...)`, `ax.plot(...)` |
| `plt.title("...")` | `ax.set_title("...")` |
| `plt.xlabel("...")`, `plt.ylabel("...")` | `ax.set_xlabel("...")`, `ax.set_ylabel("...")` |
| `plt.ylim(0, 15)` | `ax.set_ylim(0, 15)` |
| `plt.savefig("...")` | `fig.savefig("...")` |

## pandas has a shortcut

A Series or DataFrame can draw itself. `.plot.bar()` makes the figure, draws
the bars, puts the index along the bottom and names the axis after it:

In [ ]:
ax = device_counts.plot.bar(figsize=(7, 3.5))
plt.show()

print(type(ax).__name__)

It returned an `ax`, the same kind of object as before. That is the whole
trick: pandas' `.plot()` is matplotlib underneath, so you can start with the
shortcut and carry on with `ax.` calls.

The best of both: make the `ax` yourself, hand it to pandas with `ax=ax`,
then finish it by hand.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
device_counts.plot.bar(ax=ax, color="#4f6ddb", rot=0)
ax.set_title("Almost half of all plays are on the phone")
ax.set_xlabel("")
ax.set_ylabel("plays")
ax.bar_label(ax.containers[0])
plt.show()

phone_share = device_counts["phone"] / device_counts.sum()
print(f"phone: {device_counts['phone']} of {device_counts.sum()} plays, {phone_share:.1%}")
assert device_counts["phone"] == 1039
assert round(phone_share * 100, 1) == 47.6

`rot=0` keeps the labels upright; pandas turns them sideways by default.
`ax.containers[0]` is the group of bars pandas just drew, which is what
`bar_label` wants.

**When to use which.** While you are exploring, `.plot()` is perfect: one
line, a rough picture, move on. When someone else will read the chart, you
will be setting the title, the labels and the numbers on `ax` anyway, so
drop down to `ax` as soon as you start finishing it.

---

# The question first

**Which genre do I listen to longest, per play?**

You answered this in the session 4 homework with two dictionaries. In
pandas it is one line.

In [ ]:
by_genre = plays.groupby("genre")["minutes_played"].mean().sort_values(ascending=False)
by_genre

In [ ]:
fig, ax = plt.subplots()
ax.bar(by_genre.index, by_genre.values)
plt.show()

That is a chart, and it is not finished. It has no title, no units, and you
have to squint to read the values off the axis. A reader should get the
answer without you in the room.

Three things fix it:

* **a title that states the answer**, not the topic. "Minutes per genre"
  makes the reader do the work; "Jazz plays last longest" does it for them
* **axis labels with units**
* **the numbers on the bars**, rounded to something a person would say out
  loud

The rounding is the `:.1f` from session 1. "6.573571428" minutes is noise;
"6.6" is a number.

In [ ]:
top_genre = by_genre.index[0]
top_minutes = by_genre.iloc[0]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(by_genre.index, by_genre.values, color="#4f6ddb")

ax.set_title(f"{top_genre} plays last longest: {top_minutes:.1f} minutes on average")
ax.set_ylabel("average minutes per play")
ax.bar_label(bars, fmt="{:.1f}")             # the same :.1f as in an f-string
ax.spines[["top", "right"]].set_visible(False)

plt.show()

Read the chart, and the title has already told you what to look for. The
bars confirm it: Jazz at 6.6, Electronic at 5.3, everything else between
3.2 and 4.0.

`ax.bar_label` writes each bar's value on top of it. `fmt="{:.1f}"` is the
format part of an f-string, without the f and the variable. The
`spines` line removes the top and right borders, which carry no
information.

Sorting before plotting matters too. Bars in alphabetical order make the
reader hunt; bars in size order make the ranking the first thing they see.

In [ ]:
assert top_genre == "Jazz"
assert round(top_minutes, 1) == 6.6

---

# Choose the chart from the question

Almost every question you will ask of a table is one of five kinds. Say
which kind it is, and the chart picks itself:

| The question is about | Example | Chart |
|---|---|---|
| **comparing** categories | which genre lasts longest? | bar, or `barh` for long names |
| **change over time** | does listening change through the year? | line |
| **distribution**: how one number is spread | how long is a play? | histogram |
| **relationship** between two numbers | do the most-played artists get the most minutes? | scatter |
| **part of a whole** | what share of plays is each genre? | bars of percentages, or one stacked bar |

The rest of this notebook takes them in that order.

## Bar or line?

The rule is short:

* **bars for categories**: genres, devices, artists. There is no order
  between them, so nothing should connect them
* **a line for time**: months, days, years. The line says "this follows
  from that", which is only true when the x-axis has an order

Question: **does my listening change through the year?**

In [ ]:
plays["month"] = plays["played_at"].str[:7]
per_month = plays.groupby("month").size()
per_month.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(per_month.index, per_month.values, marker="o", color="#4f6ddb")

ax.set_title("Listening dips every summer, in both years")
ax.set_ylabel("plays per month")
ax.set_ylim(bottom=0)
ax.tick_params(axis="x", labelrotation=90)
ax.spines[["top", "right"]].set_visible(False)

plt.show()

Two valleys, both in June to August. The line makes the shape obvious in a
way the table of 24 numbers did not.

The months here are text, `"2024-01"`, so matplotlib treats them as 24
labels in a row and you have to turn them sideways to fit. Notebook 02 uses
real dates, and the axis gets much tidier.

`set_ylim(bottom=0)` starts the axis at zero. Without it, matplotlib zooms
in on the data, and a dip from 127 to 57 can look like falling off a cliff.
For a line you can sometimes argue for zooming in; for **bars, always start
at zero**, because the length of a bar is the number.

In [ ]:
low_months = per_month.sort_values().head(3)
print(low_months)
assert set(low_months.index) == {"2025-06", "2025-07", "2025-08"}
assert (per_month.idxmax(), per_month.max()) == ("2024-10", 127)

## A chart can mislead without one wrong number

Skip rate per device. All the numbers below are correct.

In [ ]:
skip_rate = plays.groupby("device")["skipped"].mean().sort_values(ascending=False) * 100
skip_rate

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))

left.bar(skip_rate.index, skip_rate.values, color="#c0392b")
left.set_ylim(10, 14.5)
left.set_title("Zoomed in: speaker looks terrible")

right.bar(skip_rate.index, skip_rate.values, color="#4f6ddb")
right.set_ylim(0, 15)
right.set_title("From zero: they are all much the same")
right.bar_label(right.containers[0], fmt="{:.1f}%")

plt.show()

Same data. On the left, speaker's bar is many times taller than car's; on
the right you can see the truth, 14.0% against 10.6%. Out of 336 speaker
plays, that difference is about eleven skips. It might be real, and it
might be chance. Either way it does not deserve the left-hand chart.

`plt.subplots(1, 2)` gives one row of two charts, and hands back two `ax`
objects, here called `left` and `right`. This is where the `ax.` style pays
off: every line says which chart it means.

In [ ]:
assert round(skip_rate["speaker"], 1) == 14.0
assert round(skip_rate["car"], 1) == 10.6
extra_skips = (skip_rate["speaker"] - skip_rate["car"]) / 100 * 336
print(f"{extra_skips:.1f} more skips than speaker would have at car's rate")
assert round(extra_skips) == 11

---

## Histogram: how is one number spread out?

**How long is a play?** Not the average, which is one number, but the whole
spread. A histogram cuts the minutes into ranges, called **bins**, and draws
one bar per range for how many plays fall in it. Here each bin is one minute
wide: 0 to 1, 1 to 2, and so on.

In [ ]:
minutes = plays["minutes_played"]
one_minute_bins = range(0, 13)            # edges 0, 1, 2, ... 12

middle = ((minutes >= 2) & (minutes < 6)).mean()
under_one = (minutes < 1).sum()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(minutes, bins=one_minute_bins, color="#4f6ddb", edgecolor="white")

ax.set_title(f"{middle:.0%} of plays last 2 to 6 minutes, and {under_one} last under one")
ax.set_xlabel("minutes played")
ax.set_ylabel("number of plays")
ax.spines[["top", "right"]].set_visible(False)

plt.show()

The bars touch, because the x-axis is a number with no gaps in it, not a
list of categories. Most plays sit in the middle, a long thin tail runs out
to 11 minutes, and on the far left there is a bar that is too tall for the
shape: 225 plays under one minute.

`:.0%` is one more format code: multiply by 100, no decimals, add a `%`.

That tall first bar looks like it might be the skips. Worth checking rather
than assuming. Split the plays in two and **stack** the two histograms, one
on top of the other, so each bar shows both parts.

In [ ]:
kept = plays.loc[plays["skipped"] == 0, "minutes_played"]
skips = plays.loc[plays["skipped"] == 1, "minutes_played"]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist([kept, skips], bins=one_minute_bins, stacked=True, edgecolor="white",
        color=["#4f6ddb", "#c0392b"], label=["played through", "skipped"])

ax.set_title(f"{(skips < 1).mean():.0%} of skips last under a minute")
ax.set_xlabel("minutes played")
ax.set_ylabel("number of plays")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)

plt.show()

With two colours on one chart, **a legend is not optional**: `label=` names
each part, and `ax.legend()` draws the key.

Every skip is in the first three bars; the longest lasted 2.67 minutes. Now
read two sentences that sound like the same claim:

* "most skips are short"
* "most short plays are skips"

Count them. "Short" means under one minute, the same line the session 5
stretch drew with `CASE WHEN minutes_played < 1 THEN 'short'`.

In [ ]:
short = plays["minutes_played"] < 1
skipped = plays["skipped"] == 1
both = (short & skipped).sum()

print(f"{both} plays are short and skipped")
print(f"out of {skipped.sum()} skips:      {both / skipped.sum():.1%} are short")
print(f"out of {short.sum()} short plays: {both / short.sum():.1%} are skips")

The same 190 plays, divided by two different totals. 73.6% of the 258
skips are short, and 84.4% of the 225 short plays are skips. Two different
questions, two different answers.

And where you draw the line for "short" changes which one is bigger:

In [ ]:
short_2 = plays["minutes_played"] < 2
both_2 = (short_2 & skipped).sum()

print(f"under two minutes, out of {skipped.sum()} skips:      {both_2 / skipped.sum():.1%}")
print(f"under two minutes, out of {short_2.sum()} short plays: {both_2 / short_2.sum():.1%}")

assert (both, skipped.sum(), short.sum()) == (190, 258, 225)
assert round(both / skipped.sum() * 100, 1) == 73.6
assert round(both / short.sum() * 100, 1) == 84.4
assert (both_2, short_2.sum()) == (254, 391)
assert skips.max() == 2.67
assert round(middle, 2) == 0.63

With a two-minute line, 98.4% of skips are short, but only 65.0% of short
plays are skips. "Nearly every skip is short" would be true, and "nearly
every short play is a skip" would not be, on the very same chart. Whenever a
percentage goes in a title, ask: **out of what?** The title above says
"of skips" because that is the total it divides by.

**What this chart does not show:** why a play was short. 35 plays under a
minute were not skips, and nothing in the log says what they were.

## Scatter: do two numbers move together?

**Do the artists I play most also get the most minutes?** That is two
numbers per artist, so each artist becomes one dot: plays along the bottom,
minutes up the side.

In [ ]:
per_artist = plays.groupby("artist_name").agg(
    plays=("play_id", "count"),
    minutes=("minutes_played", "sum"),
)

most_plays = per_artist["plays"].idxmax()
most_minutes = per_artist["minutes"].idxmax()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(per_artist["plays"], per_artist["minutes"], color="#4f6ddb", alpha=0.75)

for name in [most_plays, most_minutes]:
    ax.annotate(name, (per_artist.loc[name, "plays"], per_artist.loc[name, "minutes"]),
                textcoords="offset points", xytext=(8, -4), ha="left")

ax.set_title(f"{most_plays} has the most plays, {most_minutes} the most minutes")
ax.set_xlabel("plays")
ax.set_ylabel("total minutes")
ax.set_xlim(0, per_artist["plays"].max() * 1.25)     # room for the labels
ax.set_ylim(bottom=0)
ax.spines[["top", "right"]].set_visible(False)

plt.show()

per_artist.loc[[most_plays, most_minutes]].round(1)

The dots climb from bottom left to top right: more plays, more minutes, no
surprise. The interesting part is the spread around that climb. Sara
Lindqvist has 13 more plays than Glass Tram and over 400 fewer minutes,
because a Glass Tram play lasts longer: 5.1 minutes on average, against 3.2.
It is the total-versus-average lesson from the session 4 homework, drawn.
Each dot's height divided by its distance along is that artist's average
play.

`ax.annotate(text, (x, y))` writes a label next to one point.
`xytext=(8, -4)` with `textcoords="offset points"` nudges it a little to the
right, so it does not sit on top of the dot, and `ha="left"` makes the text
start there. Notebook 02 does much more with `annotate`.

**What this chart does not show:** why one artist's plays are longer. Glass
Tram is Electronic and Sara Lindqvist is Pop, and the genre averages at the
top of this notebook (5.3 and 3.2 minutes) would fit. But genre is not on
this chart, so the chart on its own cannot tell you.

In [ ]:
assert (most_plays, most_minutes) == ("Sara Lindqvist", "Glass Tram")
assert per_artist.loc[most_plays, "plays"] - per_artist.loc[most_minutes, "plays"] == 13
assert per_artist.loc[most_minutes, "minutes"] - per_artist.loc[most_plays, "minutes"] > 400

average_play = per_artist["minutes"] / per_artist["plays"]
assert round(average_play[most_minutes], 1) == 5.1
assert round(average_play[most_plays], 1) == 3.2

## Part of a whole, and why not a pie

**What share of my plays is each genre?** The shares add up to 100%, which
is what makes it a part-of-a-whole question. The chart everybody reaches for
is a pie. Try reading one:

In [ ]:
genre_share = plays["genre"].value_counts(normalize=True) * 100
ranked = genre_share.sort_values()

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.5))

left.pie(genre_share.values, labels=genre_share.index)
left.set_title("Is Latin or Afrobeat bigger?")

bars = right.barh(ranked.index, ranked.values, color="#4f6ddb")
right.bar_label(bars, fmt="{:.1f}%", padding=3)
right.set_title(f"Latin {genre_share['Latin']:.1f}%, Afrobeat {genre_share['Afrobeat']:.1f}%")
right.set_xlabel("share of all plays (%)")
right.spines[["top", "right"]].set_visible(False)

plt.show()

On the pie you are comparing angles of slices that do not start in the same
place. Nobody can do that well: Latin and Afrobeat are 185 and 180 plays,
and their slices look identical. On the bars you compare lengths from one
shared baseline, which people do well, and the labels settle it.

A pie can work for two or three slices, where the question is "is it more
than half?". Past that, use bars. Comparing a whole with another whole,
2024 against 2025, is worse still with two pies; that is one of the
homework makeovers.

`value_counts(normalize=True)` gives shares instead of counts, so the
percentages come straight out.

In [ ]:
assert round(genre_share.sum()) == 100
assert round(genre_share["Latin"], 1) == 8.5
assert round(genre_share["Afrobeat"], 1) == 8.2
assert ranked.index[-1] == "Electronic"

---

# Saving the chart

A chart in a notebook is for you. A chart in a file is for someone else.

`fig.savefig` writes it out. Save into `output/`, which git ignores, so
generated pictures never end up in a commit.

In [ ]:
os.makedirs("output", exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(by_genre.index, by_genre.values, color="#4f6ddb")
ax.set_title(f"{top_genre} plays last longest: {top_minutes:.1f} minutes on average")
ax.set_ylabel("average minutes per play")
ax.bar_label(bars, fmt="{:.1f}")
ax.spines[["top", "right"]].set_visible(False)

fig.savefig("output/minutes_by_genre.png", dpi=150, bbox_inches="tight")
plt.show()

print("saved:", Path("output/minutes_by_genre.png").exists())

Open `output/minutes_by_genre.png` from the VS Code explorer and look at it.

* `dpi=150` makes it sharp enough for a slide or a document
* `bbox_inches="tight"` trims the white border so long labels are not cut off
* **save before `plt.show()`**. In a script, `plt.savefig()` after
  `plt.show()` writes a blank picture, because pyplot has already moved on
  to a new, empty figure. Saving first never goes wrong, so make it the
  habit

## A checklist for every chart

1. Can I say the question in one sentence?
2. Is it the right kind of chart for that kind of question?
3. Does the title give the answer?
4. Do the bars start at zero?
5. Units on the axis, numbers rounded to what a person would say?
6. If there is a percentage, is it clear what it is out of?
7. What does it **not** show, and could a reader take it to?
8. Would it make sense to someone who was not in the room?

## Next

`02-time-and-panels.ipynb`: real dates, reference lines and annotations,
many charts in one figure, and highlighting the one line that matters.